![Week 1 Day 4](../assets/banners/w1d4.png)

# Week 1 Day 4 - Multimodal Inputs and Tool Calling

Yesterday you tamed the model's output: Pydantic shapes, `with_structured_output`, and a loop that sends the model its own mistakes to fix. Question 3 came back as a clean, validated object.

Today we widen what goes **in** and what the model can **do**.

First, **images**. Asha's question paper is a tidy, typed PDF, but Sam's answer booklet in week 4 will be pages of handwriting, and the only way a model can read those is to *look* at them. So today you'll render a PDF page as an image and send it to a model as part of a message.

Second, **tools**. A model can't open `expected_exam.json` or do reliable arithmetic on its own. But it can *ask you* to run a function for it. You'll write tools with `@tool`, hand them to a model with `bind_tools`, and write the tool loop by hand, so that in week 3 you'll know exactly what `create_agent` is doing for you.

![Your 6-week roadmap](../assets/roadmap/week1.png)

### 🎬 Videos for this lab
| # | Lecture |
|:-:|---|
| 1 | Day 4 - Multimodal: Sending Images and PDF Pages to a Model |
| 2 | Day 4 - Tools and Tool Calling: `@tool`, `bind_tools` and the Tool Loop |

### 🎯 By the end of this lab you will
1. 🖼️ Render a PDF page to an image, and know what it costs in tokens
2. 📨 Send it to a model as a LangChain **content block**, with or without structured output
3. 🔧 Write tools with `@tool` and see the schema the model receives
4. 🔁 Run a manual tool loop: model asks → you run the tool → you send the result back

In [ ]:
# Imports and environment first, all in one place

import base64
import json
import os

import pymupdf
from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from PIL import Image
from pydantic import BaseModel, Field

from gradewise.config import DATA_DIR, settings
from gradewise.models import chat_model, vision_model
from gradewise.schemas import ExamPaper

load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")

if openai_api_key:
    print(f"✅ OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("❌ OpenAI API Key not set. Please see Part 5 of the setup guide (and remember to SAVE .env!)")
    print("   Using another provider? That's fine: see setup/SWAP_PROVIDER.md")

print(f"🧠 Everyday model from your .env: {settings.chat_model}")

## Part 1: From PDF page to image

A PDF can give you its content in two ways:

| | **As text** (`pymupdf4llm`, used on Day 3) | **As an image** (today) |
|---|---|---|
| Works for | typed documents: question papers, schemes, course notes | handwriting, scans, diagrams, photos |
| Cost | cheap: only the words | more: you pay per *pixel tile* |
| Fidelity | perfect for typed text, loses layout and pictures | the model sees exactly what a human sees |

**PyMuPDF** (imported as `pymupdf`) renders a page to pixels. **DPI** (dots per inch) controls the resolution: 72 DPI is screen size, 200 DPI is crisp enough for handwriting.

In [ ]:
paper_pdf = DATA_DIR / "exams" / "cs_midterm" / "question_paper.pdf"

with pymupdf.open(paper_pdf) as doc:
    print(f"{paper_pdf.name}: {doc.page_count} pages")
    pix = doc[0].get_pixmap(dpi=100, alpha=False)  # ⚠️ alpha=False: transparent PDFs would render on black
    page_image = Image.frombytes("RGB", (pix.width, pix.height), pix.samples)

print(f"Rendered page 1 at 100 DPI: {page_image.width} × {page_image.height} px")
display(page_image.resize((page_image.width // 2, page_image.height // 2)))  # 💡 shown at half size

To put an image inside a message we encode it as a **data URL**: the image bytes, base64-encoded into plain text, with a small header saying what kind of image it is: `data:image/jpeg;base64,/9j/4AAQ...`. JPEG keeps it small.

In [ ]:
import io

buffer = io.BytesIO()
page_image.save(buffer, format="JPEG", quality=85)
data_url = "data:image/jpeg;base64," + base64.b64encode(buffer.getvalue()).decode("ascii")

print(f"JPEG size: {len(buffer.getvalue()) / 1024:.0f} KB → data URL of {len(data_url):,} characters")
print(data_url[:60] + "…")

### What does an image cost?

Providers don't bill images by bytes but by **pixels**. OpenAI, for example, scales the image down and cuts it into 512-pixel tiles, at roughly 170 tokens per tile plus a base cost. GradeWise has a helper for this estimate, which we borrow from week 4:

In [ ]:
from gradewise.transcription.pdf_to_images import estimate_image_tokens

with pymupdf.open(paper_pdf) as doc:
    for dpi in [72, 100, 200, 300]:
        p = doc[0].get_pixmap(dpi=dpi, alpha=False)
        img = Image.frombytes("RGB", (p.width, p.height), p.samples)
        print(f"{dpi:>3} DPI  {img.width:>4} × {img.height:<4} px  ≈ {estimate_image_tokens(img):>5,} tokens")

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/stop.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ef4444;">Stop & think</h2>
            <span style="color:#ef4444;">Look at those numbers: past a certain size, the token estimate stops growing, because the provider shrinks big images before the model sees them. So what does rendering at 300 DPI buy you? (Hint: mostly upload time.) And roughly how many tokens would a 6-page handwritten answer booklet cost, compared with the ~900 tokens of the question paper as text?</span>
        </td>
    </tr>
</table>

### Images as LangChain content blocks

In LangChain 1.x a message's `content` can be a **list of content blocks** instead of a string. Each block is a small dict with a `type`. The two we need:

```python
{"type": "text", "text": "What questions are on this page?"}
{"type": "image", "url": "data:image/jpeg;base64,..."}
```

These are **provider-neutral**. `langchain-openai` translates the image block into OpenAI's `image_url` format, `langchain-anthropic` into Claude's format, and so on. You write the message once. Let's build one, and peek at what would be sent to OpenAI (offline, nothing is sent):

In [ ]:
from langchain_core.messages import convert_to_openai_messages

page_message = HumanMessage(
    content=[
        {"type": "text", "text": "List every question number on this exam page, with its marks."},
        {"type": "image", "url": data_url},
    ]
)

print("Block types:", [block["type"] for block in page_message.content_blocks])

as_openai = convert_to_openai_messages([page_message])[0]
for part in as_openai["content"]:
    shown = {k: (str(v)[:70] + "…" if len(str(v)) > 70 else v) for k, v in part.items()}
    print(shown)

Now send it to your **vision model** (from `.env`: `VISION_MODEL`, or `CHAT_MODEL` in budget mode). A good vision model reads the page as easily as you do:

In [ ]:
vision = vision_model()

reply = vision.invoke([page_message])
display(Markdown(reply.content))
print(reply.usage_metadata)

And because an image message is still just a message, **everything from Day 3 still works**. Here we ask for a structured list of the questions on the page:

In [ ]:
class QuestionOnPage(BaseModel):
    number: str = Field(description='The question label as printed, e.g. "3" or "4(b)"')
    max_marks: float = Field(gt=0, description="The marks in square brackets")


class PageQuestions(BaseModel):
    """The questions visible on one exam page."""

    questions: list[QuestionOnPage]


page_questions = vision.with_structured_output(PageQuestions).invoke([page_message])
for q in page_questions.questions:
    print(f"Question {q.number:<6} {q.max_marks:g} marks")

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/important.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important</h2>
            <span style="color:#ff7800;">Images are the most expensive thing you'll send a model, so: render at a sensible DPI and cap the size; use JPEG, not PNG, for scans; send only the pages you need; and never send an image when the PDF's <b>text</b> would do. For typed documents like Asha's question paper, text is cheaper <i>and</i> more accurate. That's why GradeWise reads the exam pack as text, and saves vision for handwriting in week 4.</span>
        </td>
    </tr>
</table>

## Part 2: Tools and tool calling

Here's the key idea, and it surprises most people: **the model never runs your code.** Tool calling is a conversation:

![Diagram](../assets/diagrams/ba805243a997.png)

<details><summary>🔎 Diagram source (Mermaid)</summary>

```text
sequenceDiagram
    participant You as Your code
    participant LLM as Model
    You->>LLM: question + the list of tools (name, description, arguments)
    LLM-->>You: AIMessage with tool_calls: "please run get_question(question_id='Q5')"
    You->>You: run get_question("Q5")
    You->>LLM: ToolMessage with the result
    LLM-->>You: final answer (or more tool calls)
```

</details>


The model only *decides* which tool to call and with what arguments. Your code runs it. That's what makes tools safe to reason about: a tool can only do what your function does.

Let's give our model access to the exam. We load the reference answer key, `expected_exam.json`, as an `ExamPaper` (you'll meet this schema properly tomorrow):

In [ ]:
EXAM = ExamPaper.model_validate_json(
    (DATA_DIR / "exams" / "cs_midterm" / "expected_exam.json").read_text(encoding="utf-8")
)
QUESTIONS = {q.id: q for q in EXAM.questions}
print(f"{EXAM.title}: {len(QUESTIONS)} questions → {', '.join(QUESTIONS)}")

`@tool` turns a plain Python function into a LangChain tool. The **function name** becomes the tool name, the **docstring** becomes the description, and the **type hints** become the argument schema. All three are read by the model, so write them for the model.

In [ ]:
@tool
def get_question(question_id: str) -> str:
    """Look up one CS201 midterm question by its ID (e.g. "Q3", "Q7b"): returns its text and max marks."""
    q = QUESTIONS.get(question_id.strip())
    if q is None:  # 💡 a helpful error lets the model correct itself
        return f"No question {question_id!r}. Valid IDs: {', '.join(QUESTIONS)}"
    return json.dumps({"id": q.id, "text": q.text, "max_marks": q.max_marks})


@tool
def calculate_percentage(marks_scored: float, marks_available: float) -> float:
    """Turn a score into a percentage, rounded to 1 decimal place. Use this instead of doing arithmetic yourself."""
    return round(100 * marks_scored / marks_available, 1)


tools = [get_question, calculate_percentage]

# 🆓 Tools are ordinary Runnables: you can test them directly, no model needed
print(get_question.invoke({"question_id": "Q7b"}))
print(calculate_percentage.invoke({"marks_scored": 7, "marks_available": 9}))

In [ ]:
from langchain_core.utils.function_calling import convert_to_openai_tool

# 💡 This is exactly what the model is told about the tool
print(json.dumps(convert_to_openai_tool(get_question), indent=2))

### `bind_tools`: offering the tools to the model

`model.bind_tools(tools)` returns a model that sends the tool schemas along with every call. When the model wants a tool, its reply has **empty content** and a list of `tool_calls`:

In [ ]:
llm = chat_model(temperature=0)
llm_with_tools = llm.bind_tools(tools)

ai = llm_with_tools.invoke("How many marks is question 7(b) worth?")
print("content   :", repr(ai.content))
print("tool_calls:", ai.tool_calls)

### The tool loop, by hand

A single request may need several tools, one after another. So we loop:

1. Send the messages to the model.
2. If the reply has no `tool_calls`, it's the final answer: stop.
3. Otherwise run each tool, append the model's request **and** a `ToolMessage` per result, and go back to 1.

Two safety rails: a **turn limit**, so a confused model can't loop for ever, and a check for **unknown tool names**.

In [ ]:
def run_tool_loop(model, question: str, tools: list, max_turns: int = 5) -> str:
    """Let `model` call `tools` until it gives a final answer."""
    by_name = {t.name: t for t in tools}
    model_with_tools = model.bind_tools(tools)
    messages = [
        SystemMessage("You answer questions about the CS201 midterm. Use the tools; never guess marks."),
        HumanMessage(question),
    ]
    for turn in range(1, max_turns + 1):
        ai = model_with_tools.invoke(messages)
        messages.append(ai)  # ⚠️ the model's tool request must stay in the history, before the results
        if not ai.tool_calls:
            return ai.content
        for call in ai.tool_calls:
            print(f"🔧 turn {turn}: {call['name']}({call['args']})")
            if call["name"] in by_name:
                result = by_name[call["name"]].invoke(call)  # 💡 invoking with the call returns a ToolMessage
            else:
                result = ToolMessage(f"Unknown tool {call['name']!r}", tool_call_id=call["id"])
            print(f"   ↳ {result.content}")
            messages.append(result)
    return f"🙋 Gave up after {max_turns} turns"

### Dress rehearsal with a scripted model

Before spending money, let's run the loop against a **scripted** fake model. We hand it the exact replies a real model would give: two lookups, a calculation, then the answer. (The fake doesn't really read the tools, so we give it a do-nothing `bind_tools`.)

In [ ]:
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel
from langchain_core.messages import AIMessage


class ScriptedModel(GenericFakeChatModel):
    """Replies with the next scripted message. bind_tools is a no-op, so it fits our loop."""

    def bind_tools(self, tools, **kwargs):
        return self


def call(name: str, call_id: str, **args) -> dict:
    return {"name": name, "args": args, "id": call_id}


script = [
    AIMessage(
        "", tool_calls=[call("get_question", "c1", question_id="Q3"), call("get_question", "c2", question_id="Q5")]
    ),
    AIMessage("", tool_calls=[call("calculate_percentage", "c3", marks_scored=7, marks_available=9)]),
    AIMessage("Sam scored 7 of the 9 marks available on Q3 and Q5: 77.8%."),
]

question = "Sam scored 4 on Q3 and 3 on Q5. What percentage of the marks available on those two questions is that?"
print("💬", run_tool_loop(ScriptedModel(messages=iter(script)), question, tools))

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/stop.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ef4444;">Stop & think</h2>
            <span style="color:#ef4444;">The rehearsal shows the loop's plumbing works. Before running the real model below, predict: will it call <code>get_question</code> twice in the <i>same</i> turn (in parallel) or in two turns? Will it use <code>calculate_percentage</code>, or do the sum itself despite the docstring?</span>
        </td>
    </tr>
</table>

In [ ]:
print("💬", run_tool_loop(llm, question, tools))

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/gradewise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#7c5cff;">GradeWise</h2>
            <span style="color:#7c5cff;">You've now built, by hand, the loop at the heart of every agent. In <b>week 3</b>, <code>langchain.agents.create_agent</code> runs this same loop for you (as a LangGraph graph you can draw). In <b>week 5</b> GradeWise's grader gets tools just like these: <code>get_rubric</code>, <code>retrieve_reference</code> and <code>flag_for_review</code>. And the image half of today lives in <code>gradewise/transcription/pdf_to_images.py</code> (<code>render_pages</code>, <code>to_data_url</code>) and <code>gradewise/transcription/vision.py</code>, which sends exactly the content blocks you built.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/business.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#22c55e;">Business value</h2>
            <span style="color:#22c55e;">Tools are how an LLM touches real systems: a student database, a timetable, a grade book. Because <i>your</i> code runs every tool, you decide what the model can and can't do. A read-only <code>get_question</code> tool is harmless; a <code>delete_student</code> tool would need a human to approve it, which is exactly what week 3's human-in-the-loop is for.</span>
        </td>
    </tr>
</table>

## Recap, and where we are heading

Today you:
- 🖼️ rendered a PDF page with `pymupdf`, encoded it as a data URL, and estimated its token cost at different DPIs
- 📨 sent it as provider-neutral content blocks (`{"type": "image", "url": ...}`), and combined vision with structured output
- 🔧 wrote tools with `@tool`, tested them directly, and read the schema the model receives
- 🔁 wrote the tool loop by hand, rehearsed it with a scripted model, then ran it for real

You now have every ingredient for this week's milestone: prompts, Pydantic, structured output, PDFs. **Tomorrow (Day 5) is project day 🧱.** We tour GradeWise's architecture, design the full exam schema, and turn Asha's entire exam pack into one validated JSON file, then check it field by field against the answer key.

⏭ **Next:** `week1/day5.ipynb`

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/exercise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f59e0b;">Exercise</h2>
            <span style="color:#f59e0b;"><b>A third tool.</b> Add a <code>list_questions(question_type: str)</code> tool that returns the IDs and marks of every question of that type (<code>"numerical"</code>, <code>"definition"</code>, ...), with a helpful message for an unknown type. Then ask: <i>"Which numerical questions are on the paper, and what percentage of the whole paper are they worth?"</i> Rehearse with <code>ScriptedModel</code> first, then run the real model. Bonus: send page 2 of the question paper as an image and have the model check the marks it reads using <code>get_question</code>. Solution in <code>week1/solutions/day4_SOLUTION.ipynb</code>.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/resources.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#3b82f6;">Resources</h2>
            <span style="color:#3b82f6;">• Multimodal messages: <a href="https://docs.langchain.com/oss/python/langchain/messages">docs.langchain.com/oss/python/langchain/messages</a><br/> • Tools: <a href="https://docs.langchain.com/oss/python/langchain/tools">docs.langchain.com/oss/python/langchain/tools</a><br/> • PyMuPDF rendering: <a href="https://pymupdf.readthedocs.io/en/latest/page.html#Page.get_pixmap">pymupdf.readthedocs.io</a></span>
        </td>
    </tr>
</table>